# Preprocessing CBIS-DDSM dataset

# Important libraries

# Preprocessing class with all needed functions

-------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
pip install pydicom

In [1]:
import os
import pydicom
import numpy as np
import pandas as pd
import cv2

# Calc Test Loading & Transforming to PNG format (simple processing)

In [2]:
# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_test_set_png"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/calc_case_description_test_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    """
    Determine if the folder contains full mammogram, ROI, or mask images
    based on the folder description in the path
    """
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"  # This folder contains both ROI and mask
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row['patient_id']
    
    # Process each path type
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue
            
        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)
        
        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()  # Ensure consistent ordering
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        # Determine the image type from folder description
        folder_type = get_image_type(file_location)
        
        # Create patient subfolder
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_id)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_id)
        else:  # mask
            patient_save_dir = os.path.join(MASK_DIR, patient_id)
            
        os.makedirs(patient_save_dir, exist_ok=True)

        # Process files based on type
        if folder_type == "full":
            # Full mammogram - process all files
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dcm_path = os.path.join(full_file_dir, dcm_file)
                
                dicom_to_png(dcm_path, save_path)
                    
        elif folder_type == "roi_mask":
            # ROI/Mask folder - separate based on file naming
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                
                # Check if this is ROI (1-1) or mask (1-2) based on filename
                if dcm_file.endswith("1-1.dcm"):
                    # This is ROI
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                        os.makedirs(os.path.join(ROI_CROP_DIR, patient_id), exist_ok=True)
                        
                        dicom_to_png(dcm_path, save_path)
                            
                elif dcm_file.endswith("1-2.dcm"):
                    # This is mask
                    if path_type == "mask":
                        output_name = f"{subject_id}_mask.png"
                        save_path = os.path.join(MASK_DIR, patient_id, output_name)
                        os.makedirs(os.path.join(MASK_DIR, patient_id), exist_ok=True)
                        
                        dicom_to_png(dcm_path, save_path)

print("[INFO] Processing completed!")


[INFO] Processing completed!


# Calc Training Loading & Transforming to PNG format (complex processing)

In [4]:
# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/calc_case_description_train_set_png"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/calc_case_description_train_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    """
    Determine if the folder contains full mammogram, ROI, or mask images
    based on the folder description in the path
    """
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"  # This folder contains both ROI and mask
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row['patient_id']
    
    # Process each path type
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue
            
        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)
        
        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()  # Ensure consistent ordering
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        # Determine the image type from folder description
        folder_type = get_image_type(file_location)
        
        # Create patient subfolder
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_id)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_id)
        else:  # mask
            patient_save_dir = os.path.join(MASK_DIR, patient_id)
            
        os.makedirs(patient_save_dir, exist_ok=True)

        # Process files based on type
        if folder_type == "full":
            # Full mammogram - process all files
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dcm_path = os.path.join(full_file_dir, dcm_file)

                dicom_to_png(dcm_path, save_path)
                    
        elif folder_type in ["roi_mask", "roi", "mask"]:
            # Handle ROI, mask, or mixed folder cases
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()
        
                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                        os.makedirs(os.path.dirname(save_path), exist_ok=True)
                        dicom_to_png(dcm_path, save_path)
                        continue
        
                # --- MASK (ROI mask images) ---
                if path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
                    continue
        
                # --- Fallback (old naming 1-1 / 1-2) ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
        
                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)



print("[INFO] Processing completed!")


[INFO] Processing completed!


# Mass Test Loading & Transforming to PNG format (complex processing)

In [ ]:
# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/mass_case_description_test_set_png"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/mass_case_description_test_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    """
    Determine if the folder contains full mammogram, ROI, or mask images
    based on the folder description in the path
    """
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"  # This folder contains both ROI and mask
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row['patient_id']
    
    # Process each path type
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue
            
        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)
        
        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()  # Ensure consistent ordering
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        # Determine the image type from folder description
        folder_type = get_image_type(file_location)
        
        # Create patient subfolder
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_id)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_id)
        else:  # mask
            patient_save_dir = os.path.join(MASK_DIR, patient_id)
            
        os.makedirs(patient_save_dir, exist_ok=True)

        # Process files based on type
        if folder_type == "full":
            # Full mammogram - process all files
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dcm_path = os.path.join(full_file_dir, dcm_file)

                dicom_to_png(dcm_path, save_path)
                    
        elif folder_type in ["roi_mask", "roi", "mask"]:
            # Handle ROI, mask, or mixed folder cases
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()
        
                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                        os.makedirs(os.path.dirname(save_path), exist_ok=True)
                        dicom_to_png(dcm_path, save_path)
                        continue
        
                # --- MASK (ROI mask images) ---
                if path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
                    continue
        
                # --- Fallback (old naming 1-1 / 1-2) ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
        
                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)



print("[INFO] Processing completed!")


# Mass Training Loading & Transforming to PNG format (complex processing)

In [3]:
# === CONFIG ===
DICOM_ROOT = "D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/CBIS-DDSM"
OUTPUT_ROOT = "D:/cbis-ddsm_dataset_licenta/data/processed_separate_roi_full_masks/mass_case_description_train_set_png"

FULL_IMG_DIR = os.path.join(OUTPUT_ROOT, "full_images")
MASK_DIR = os.path.join(OUTPUT_ROOT, "masks")
ROI_CROP_DIR = os.path.join(OUTPUT_ROOT, "roi_crops")

os.makedirs(FULL_IMG_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ROI_CROP_DIR, exist_ok=True)

# === Load CSVs ===
test_set_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/mass_case_description_train_set.csv")
metadata_df = pd.read_csv("D:/cbis-ddsm_dataset_licenta/data/raw/manifest-ZkhPvrLo5216730872708713142/metadata.csv")

# Normalize paths
metadata_df["File Location"] = (
    metadata_df["File Location"]
    .str.replace("\\", "/")
    .str.strip("./")
    .str.replace("^CBIS-DDSM/", "", regex=True)
)
test_set_df = test_set_df.rename(columns={
    "image file path": "full_path",
    "cropped image file path": "roi_path",
    "ROI mask file path": "mask_path"
})

# === Helper: Convert DICOM to PNG ===
def dicom_to_png(dicom_path, save_path):
    try:
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)

        pixel_array -= pixel_array.min()
        if pixel_array.max() != 0:
            pixel_array /= pixel_array.max()
        pixel_array *= 255.0
        img = pixel_array.astype(np.uint8)

        cv2.imwrite(save_path, img)
        return True
    except Exception as e:
        print(f"[ERROR] Failed processing {dicom_path}: {e}")
        return False

# === Helper: Determine image type based on folder description ===
def get_image_type(file_location):
    """
    Determine if the folder contains full mammogram, ROI, or mask images
    based on the folder description in the path
    """
    if "full mammogram images" in file_location:
        return "full"
    elif "ROI mask images" in file_location:
        return "roi_mask"  # This folder contains both ROI and mask
    elif "cropped images" in file_location:
        return "roi"
    else:
        return "unknown"

# === Process Each Entry ===
for idx, row in test_set_df.iterrows():
    patient_id = row['patient_id']
    
    # Process each path type
    for path_type, csv_column in [("full", "full_path"), ("roi", "roi_path"), ("mask", "mask_path")]:
        if pd.isna(row[csv_column]):
            continue
            
        subject_id = row[csv_column].split("/")[0]
        match = metadata_df[metadata_df["Subject ID"] == subject_id]

        if match.empty:
            print(f"[WARNING] No match for {subject_id} in metadata.")
            continue

        file_location = match["File Location"].values[0]
        full_file_dir = os.path.join(DICOM_ROOT, file_location)
        
        try:
            dcm_files = [f for f in os.listdir(full_file_dir) if f.endswith(".dcm")]
            dcm_files.sort()  # Ensure consistent ordering
        except FileNotFoundError:
            print(f"[ERROR] Directory not found: {full_file_dir}")
            continue

        # Determine the image type from folder description
        folder_type = get_image_type(file_location)
        
        # Create patient subfolder
        if path_type == "full":
            patient_save_dir = os.path.join(FULL_IMG_DIR, patient_id)
        elif path_type == "roi":
            patient_save_dir = os.path.join(ROI_CROP_DIR, patient_id)
        else:  # mask
            patient_save_dir = os.path.join(MASK_DIR, patient_id)
            
        os.makedirs(patient_save_dir, exist_ok=True)

        # Process files based on type
        if folder_type == "full":
            # Full mammogram - process all files
            for dcm_file in dcm_files:
                output_name = f"{subject_id}_{dcm_file.replace('.dcm', '.png')}"
                save_path = os.path.join(patient_save_dir, output_name)
                dcm_path = os.path.join(full_file_dir, dcm_file)

                dicom_to_png(dcm_path, save_path)
                    
        elif folder_type in ["roi_mask", "roi", "mask"]:
            # Handle ROI, mask, or mixed folder cases
            for dcm_file in dcm_files:
                dcm_path = os.path.join(full_file_dir, dcm_file)
                folder_lower = file_location.lower()
        
                # --- ROI (cropped images) ---
                if any(kw in folder_lower for kw in ["cropped images", "cropped_image", "cropped"]):
                    if path_type == "roi":
                        output_name = f"{subject_id}_roi.png"
                        save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                        os.makedirs(os.path.dirname(save_path), exist_ok=True)
                        dicom_to_png(dcm_path, save_path)
                        continue
        
                # --- MASK (ROI mask images) ---
                if path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
                    continue
        
                # --- Fallback (old naming 1-1 / 1-2) ---
                if dcm_file.endswith("1-1.dcm") and path_type == "roi":
                    output_name = f"{subject_id}_roi.png"
                    save_path = os.path.join(ROI_CROP_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)
        
                elif dcm_file.endswith("1-2.dcm") and path_type == "mask":
                    output_name = f"{subject_id}_mask.png"
                    save_path = os.path.join(MASK_DIR, patient_id, output_name)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    dicom_to_png(dcm_path, save_path)



print("[INFO] Processing completed!")


[INFO] Processing completed!
